In [2]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.4 MB/s eta 0:00:0000:01


In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
import os  

def find_file(filename, search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

print("Searching for dataset files...")
features_path = find_file('elliptic_txs_features.csv')
classes_path = find_file('elliptic_txs_classes.csv')
edgelist_path = find_file('elliptic_txs_edgelist.csv')

if not (features_path and classes_path and edgelist_path):
    raise FileNotFoundError(" ΤΑ ΑΡΧΕΙΑ ΔΕΝ ΒΡΕΘΗΚΑΝ! Βεβαιώσου ότι πάτησες 'Add Input' και πρόσθεσες το 'Elliptic Data Set'.")

print(f" Found Features: {features_path}")
print(f" Found Classes:  {classes_path}")
print(f" Found Edgelist: {edgelist_path}")


def load_elliptic(features_path, classes_path, edgelist_path):
    features = pd.read_csv(features_path, header=None)
    classes = pd.read_csv(classes_path)
    edges = pd.read_csv(edgelist_path)

    mapping = {'unknown': -1, '1': 1, '2': 0, 'illicit': 1, 'licit': 0}
    classes['class'] = classes['class'].map(mapping).fillna(-1).astype(int)

    node_ids = features[0].values
    id_map = {j: i for i, j in enumerate(node_ids)}
    edges['txId1'] = edges['txId1'].map(id_map)
    edges['txId2'] = edges['txId2'].map(id_map)
    edges = edges.dropna().astype(int)

    x = torch.tensor(features.iloc[:, 1:].values, dtype=torch.float)
    y = torch.tensor(classes['class'].values, dtype=torch.long)
    edge_index = torch.tensor(edges.values.T, dtype=torch.long)

    timestep = features.iloc[:, 1].values
    class_values = classes['class'].values

    train_mask = (timestep <= 34) & (class_values != -1)
    test_mask = (timestep > 34) & (class_values != -1)

    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = torch.tensor(train_mask, dtype=torch.bool)
    data.test_mask = torch.tensor(test_mask, dtype=torch.bool)
    
    return data

class StandardGCN(torch.nn.Module):
    def __init__(self, num_features, hidden_channels, num_classes):
        super(StandardGCN, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

def run_experiment(data, epochs=400):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")
    
    data = data.to(device)
    
    model = StandardGCN(num_features=166, hidden_channels=128, num_classes=2).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    
    weights = torch.tensor([1.0, 5.0]).to(device)
    criterion = torch.nn.NLLLoss(weight=weights)

    print("Training Standard GCN...")
    
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        if epoch % 50 == 0:
            print(f'Epoch {epoch:03d}, Loss: {loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        
        probs = torch.exp(out)[:, 1]
        
        y_true = data.y[data.test_mask].cpu().numpy()
        probs_test = probs[data.test_mask].cpu().numpy()
        
        best_f1 = 0
        best_thresh = 0.5
        best_precision = 0
        best_recall = 0
        
        thresholds = np.arange(0.1, 0.9, 0.05)
        for thresh in thresholds:
            y_pred_t = (probs_test > thresh).astype(int)
            score = f1_score(y_true, y_pred_t, pos_label=1, average='binary')
            
            if score > best_f1:
                best_f1 = score
                best_thresh = thresh
                best_precision = precision_score(y_true, y_pred_t, pos_label=1, average='binary', zero_division=0)
                best_recall = recall_score(y_true, y_pred_t, pos_label=1, average='binary', zero_division=0)
        
    
    print(f"FINAL RESULTS (GCN - Temporal Split 35-49)")
    print("-" * 40)
    print(f"Best Threshold Found: {best_thresh:.2f}")
    print(f"Illicit F1-Score:     {best_f1:.4f}")
    print(f"Precision:            {best_precision:.4f}")
    print(f"Recall:               {best_recall:.4f}")
    print("-" * 40)


data = load_elliptic(features_path, classes_path, edgelist_path)
run_experiment(data, epochs=400)

Searching for dataset files...
 Found Features: /kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_features.csv
 Found Classes:  /kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_classes.csv
 Found Edgelist: /kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv
Running on device: cuda
Training Standard GCN...
Epoch 050, Loss: 0.3743
Epoch 100, Loss: 0.2958
Epoch 150, Loss: 0.2551
Epoch 200, Loss: 0.2340
Epoch 250, Loss: 0.2194
Epoch 300, Loss: 0.2087
Epoch 350, Loss: 0.1964
Epoch 400, Loss: 0.1916
FINAL RESULTS (GCN - Temporal Split 35-49)
----------------------------------------
Best Threshold Found: 0.65
Illicit F1-Score:     0.4948
Precision:            0.5706
Recall:               0.4367
----------------------------------------
